# Étape 4 : Vector Store (Chroma)

- Objectif : stocker les embeddings dans une base vectorielle locale (Chroma),
puis faire de la recherche par similarité.

On va :
1) Charger un texte (speech.txt)
2) Découper en chunks
3) Calculer les embeddings (Ollama)
4) Indexer dans Chroma
5) Interroger : similarity_search / similarity_search_with_score
6) Sauvegarder sur disque et recharger
7) Utiliser le mode Retriever

## - Requirements (Chroma)

Dépendances Python :
- chromadb
- langchain
- langchain-community
- langchain-text-splitters
- langchain-chroma (intégration officielle)

Pré-requis système :
- Ollama installé et lancé
- Modèle embeddings téléchargé (recommandé : mxbai-embed-large)



In [ ]:
%pip install -U chromadb langchain langchain-community langchain-text-splitters langchain-chroma


### 1. Construire une base vectorielle (in-memory)
---

- 📌 Pipeline :
TextLoader → RecursiveCharacterTextSplitter → OllamaEmbeddings → Chroma.from_documents()


In [ ]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader = TextLoader("speech.txt", encoding="utf-8")
data = loader.load()

print("Docs chargés :", len(data))
print("Extrait :", data[0].page_content[:200])


text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

print("Chunks créés :", len(splits))
print("Extrait chunk 0 :", splits[0].page_content[:200])

embedding = OllamaEmbeddings(model="mxbai-embed-large")

vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

print("✅ VectorDB créée :", vectordb)

### 2. Interroger la base (similarity_search)
--- 

`similarity_search` retourne une liste de Documents pertinents.


In [ ]:
query = "What does the speaker believe is the main reason the United States should enter the war?"
docs = vectordb.similarity_search(query, k=3)

print("Nombre de résultats :", len(docs))
print("\n--- Top 1 ---\n", docs[0].page_content[:600])
print("\nMetadata Top 1 :", docs[0].metadata)


### 3. Sauvegarder sur disque (persist_directory)
---

On persiste la base dans un dossier local `./chroma_db`.
Utile pour éviter de recalculer à chaque exécution.

In [ ]:
persist_dir = "./chroma_db"

vectordb_persisted = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_dir
)

print("✅ VectorDB persistée dans :", persist_dir)

### 4. Recharger depuis disque
---

On recharge avec :
Chroma(persist_directory=..., embedding_function=...)

In [ ]:
db2 = Chroma(persist_directory=persist_dir, embedding_function=embedding)

docs2 = db2.similarity_search(query, k=1)

print("✅ Requête sur DB rechargée OK")
print(docs2[0].page_content[:600])

### 5. Similarity search avec score
--- 

- `similarity_search_with_score` retourne :
[(Document, score), ...]

- `Interprétation` :
    - Selon backend, plus petit score peut être "plus proche" (distance)
    - ou score plus grand peut être "meilleur" (similarity)
    -  On vérifie empiriquement en regardant l’ordre des résultats.


In [ ]:
docs_scores = vectordb.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(docs_scores, start=1):
    print(f"\n--- Résultat {i} | Score: {score} ---")
    print(doc.page_content[:350])

### 6. Mode Retriever (recommandé en RAG)
---

- Un retriever est une interface standard qui “récupère” des documents pertinents.
- C’est généralement ce qu’on branche dans une RetrievalChain.

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(query)

print("Nombre de documents récupérés :", len(retrieved))
print("\n--- Top 1 retriever ---\n", retrieved[0].page_content[:600])

## Take Away (Chroma)

- Chroma stocke des vecteurs + leurs documents associés
- `Chroma.from_documents(...)` crée et indexe directement
- `similarity_search(query)` récupère les chunks pertinents
- `persist_directory` permet de sauvegarder / recharger
- `as_retriever()` est l’interface standard utilisée dans les pipelines RAG

🎯 Prochaine étape logique :
Construire une mini chaîne RAG :
Retriever → Prompt → LLM → Réponse
(100% local avec Ollama)
